# 06 — Optional extension: temporal network comparison

Build **separate conceptual networks** for sequential slices (policy PDF by page range, or Yelp reviews by year), then compare overlap (Jaccard, node/edge churn). Aligns with the client brief *optional extension*: temporal comparisons across documents.

**Outputs** (under `../results/`): `temporal_policy_slice_summary.csv`, `temporal_policy_pairwise.csv`, and when Yelp data is available, `temporal_yelp_*.csv`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, '..')

RESULTS = Path('../results')
RESULTS.mkdir(parents=True, exist_ok=True)

## 1. Policy document: two halves (by page)

Uses a lower `min_freq` per slice so sparse segments still yield a graph.

In [ ]:
from src.ingest.pdf_reader import read_policy_pdf
from src.preprocessing.tokeniser import SpacyTokeniser
from src.extraction.concept_extractor import ConceptExtractor
from src.extensions.temporal import TemporalAnalyser
from src.extensions.temporal_slicing import policy_pages_to_temporal_slices
from src.network.graph_builder import GraphBuilder

pages = read_policy_pdf()
tokeniser = SpacyTokeniser()
policy_slices = policy_pages_to_temporal_slices(pages, tokeniser, n_chunks=2)

ce = ConceptExtractor(min_freq=1)
ta = TemporalAnalyser(concept_extractor=ce, graph_builder=GraphBuilder(min_edge_weight=1))
policy_temporal = ta.build_slices(policy_slices)

print(ta.slice_summary(policy_temporal).to_string(index=False))
print('\nConsecutive slice comparison:')
print(ta.comparison_table(policy_temporal).to_string(index=False))

ta.slice_summary(policy_temporal).to_csv(RESULTS / 'temporal_policy_slice_summary.csv', index=False)
ta.comparison_table(policy_temporal).to_csv(RESULTS / 'temporal_policy_pairwise.csv', index=False)

## 2. Yelp (optional): networks by review year

Requires extracted Yelp JSONL under `data/Yelp JSON/yelp_dataset/`. Skip this section if files are missing.

In [ ]:
from src.extensions.temporal_slicing import yelp_reviews_to_year_slices
from src.ingest.yelp_reader import load_yelp_reviews
from src.network.graph_builder import GraphBuilder

try:
    ydf = load_yelp_reviews(category_filter='Restaurants', sample_n=800)
    yelp_slices = yelp_reviews_to_year_slices(ydf, tokeniser, sample_per_year=120, min_reviews=5)
    if len(yelp_slices) >= 2:
        yta = TemporalAnalyser(concept_extractor=ce, graph_builder=GraphBuilder(min_edge_weight=1))
        y_temporal = yta.build_slices(yelp_slices)
        print(yta.slice_summary(y_temporal).to_string(index=False))
        print(yta.comparison_table(y_temporal).to_string(index=False))
        yta.slice_summary(y_temporal).to_csv(RESULTS / 'temporal_yelp_slice_summary.csv', index=False)
        yta.comparison_table(y_temporal).to_csv(RESULTS / 'temporal_yelp_pairwise.csv', index=False)
    else:
        print('Not enough year bins with reviews; increase sample_n or lower min_reviews.')
except FileNotFoundError as e:
    print('Yelp data not available:', e)

## 3. Interpretation (for reports)

- **Jaccard nodes/edges**: similarity between adjacent slices; low values suggest vocabulary or thematic shift.
- **Policy**: differences often reflect section focus (e.g. governance vs. local action) rather than calendar time.
- **Yelp**: year bins approximate cultural/language drift only weakly; interpret with caution.
- **Limitations**: each slice re-runs extraction independently; thresholds and sampling strongly affect comparability.